# 🌲 Random Forest com tidymodels e ranger

**Objetivo:** Este notebook ensina como construir, treinar e avaliar um modelo de Random Forest para classificação usando o framework tidymodels em R.

---

### O que é Random Forest?

Random Forest é um algoritmo de **ensemble learning** que constrói múltiplas árvores de decisão durante o treinamento e retorna a classe que é a **moda** das classes preditas por cada árvore individual.

**Vantagens:**
- Resistente a overfitting
- Funciona bem com dados desbalanceados
- Não requer normalização dos dados
- Lida bem com valores ausentes

**Aplicação:** Classificação de risco de crédito (nosso exemplo)

![Random Forest Schema](https://upload.wikimedia.org/wikipedia/commons/5/56/Random_forest_schema.png)

---

## 1. Carregamento de Bibliotecas

Primeiro, carregamos todas as bibliotecas necessárias. O tidymodels é um metapacote que abrange:
- **parsnip**: sintaxe uniforme para modelos
- **recipes**: pré-processamento de dados
- **rsample**: divisão de dados e validação cruzada
- **yardstick**: métricas de avaliação
- **tune**: ajuste de hiperparâmetros

In [ ]:
# Instalação (descomente se necessário)
# install.packages(c("tidymodels", "ranger", "vip", "palmerpenguins", "ggplot2", "dplyr"))

# Carregamento de todas as bibliotecas
library(tidymodels)      # Framework completo para modelagem
library(ranger)         # Motor Random Forest (rápido, em C++)
library(vip)            # Visualização de importância de variáveis
library(palmerpenguins) # Dataset de exemplo (alternativa ao German Credit)
library(ggplot2)        # Visualização
library(dplyr)         # Manipulação de dados
library(readr)         # Leitura de arquivos

# Verificar versões
tidymodels_prefer()  # Resolve conflitos de funções

---

## 2. Carregamento de Dados

Usaremos o dataset **Palmer Penguins** como alternativa moderna ao German Credit Data. Este conjunto contém dados sobre pinguins da estação Palmer, Antártica.

**Problema:** Classificar a espécie do pinguim (Adelie, Chinstrap ou Gentoo) com base em medidas físicas.

Este dataset é ideal para ensino porque:
- É pequeno e carrega rápido
- Tem variáveis categóricas e numéricas
- Contém valores ausentes (realidade de dados reais)
- Tem um número balanceado de classes

In [ ]:
# Carregar o dataset Palmer Penguins (já vem com o pacote palmerpenguins)
data(penguins)

# Ver a estrutura dos dados
str(penguins)

# Resumo estatístico
summary(penguins)

---

## 3. Análise Exploratória

Antes de modelar, precisamos conhecer nossos dados. Vamos verificar:
- **Dimensões:** quantas linhas e colunas?
- **Tipos de variáveis:** categóricas vs numéricas
- **Valores ausentes:** quantos e onde?
- **Distribuição das classes:** está balanceada?

In [ ]:
# Dimensões do dataset
cat("Dimensões:", dim(penguins), "\n")

# Verificar valores ausentes
cat("\nValores ausentes por coluna:\n")
print(colSums(is.na(penguins)))

# Contagem da variável alvo (species)
cat("\nDistribuição das espécies:\n")
print(table(penguins$species))

# Percentual por classe
cat("\nPercentual:\n")
print(round(prop.table(table(penguins$species)) * 100, 2))

In [ ]:
# Visualização: distribuição das espécies
ggplot(penguins, aes(x = species, fill = species)) +
  geom_bar() +
  labs(
    title = "Distribuição das Espécies de Pinguins",
    x = "Espécie",
    y = "Contagem",
    fill = "Espécie"
  ) +
  theme_minimal() +
  theme(legend.position = "none")

In [ ]:
# Visualização: relação entre variáveis numéricas por espécie
ggplot(penguins, aes(x = bill_length_mm, y = bill_depth_mm, color = species)) +
  geom_point(alpha = 0.7, size = 2) +
  labs(
    title = "Comprimento vs Profundidade do Bico por Espécie",
    x = "Comprimento do bico (mm)",
    y = "Profundidade do bico (mm)",
    color = "Espécie"
  ) +
  theme_minimal()

---

## 4. Pré-processamento com recipes

O pacote **recipes** do tidymodels define um pipeline de pré-processamento que:
- Transforma variáveis numéricas
- Codifica variáveis categóricas (one-hot encoding)
- Trata valores ausentes
- Normaliza dados

**Importante:** Todas as transformações são aprendidas apenas nos dados de treino e depois aplicadas ao teste — evitando data leakage!

In [ ]:
# Criar a recipe: definir a fórmula e os passos de pré-processamento
recipe_penguins <- recipe(species ~ ., data = penguins) |>
  # Passo 1: remover variáveis não utilizadas (island e year não usaremos)
  update_roles(c(island, year), new_role = "skip") |>
  
  # Passo 2: imputar valores ausentes na variável numérica (mediana)
  step_impute_median(all_numeric_predictors()) |>
  
  # Passo 3: criar dummies para variáveis categóricas (sex)
  step_dummy(all_nominal_predictors())

# Ver a recipe
recipe_penguins

---

## 5. Divisão Train/Test

Separamos os dados em **treino** (para ajustar o modelo) e **teste** (para avaliar o desempenho real).

Usaremos 75% para treino e 25% para teste, com estratificação pela variável alvo para manter a proporção de classes.

In [ ]:
# Definir a semente para reprodutibilidade
set.seed(42)

# Dividir os dados: 75% treino, 25% teste
split_penguins <- initial_split(penguins, prop = 0.75, strata = species)

# Extrair os conjuntos
train_penguins <- training(split_penguins)
test_penguins <- testing(split_penguins)

# Verificar dimensões
cat("Conjunto de treino:", nrow(train_penguins), "amostras\n")
cat("Conjunto de teste:", nrow(test_penguins), "amostras\n")

# Verificar distribuição no treino
cat("\nDistribuição no treino:\n")
print(table(train_penguins$species))

---

## 6. Definição do Modelo Random Forest

No tidymodels, definimos o modelo em duas etapas:

1. **engine**: qual implementação usar (aqui, ranger)
2. **mode**: classificação ou regressão

**Hiperparâmetros principais do Random Forest:**
- `trees`: número de árvores na floresta (mais = mais lento, mas geralmente melhor)
- `mtry`: número de variáveis sorteadas em cada divisão
- `min_n`: número mínimo de observações para dividir um nó

In [ ]:
# Definir o modelo Random Forest com parsnip
model_rf <- rand_forest(
  trees = 500,        # 500 árvores
  mtry = 3,           # 3 variáveis sorteadas em cada split
  min_n = 20          # mínimo 20 obs por folha
) |>
  set_engine("ranger") |>      # Usar ranger como motor
  set_mode("classification")   # Problema de classificação

# Ver o modelo
model_rf

---

## 7. Treinamento do Modelo

Agora vamos **preparar os dados** com a recipe e depois **treinar** o modelo.

O workflow do tidymodels combina recipe + model em um único pipeline.

In [ ]:
# Criar o workflow (pipeline completo)
wf_penguins <- workflow() |>
  add_recipe(recipe_penguins) |>
  add_model(model_rf)

# Treinar (fit) o modelo no conjunto de treino
cat("Treinando o modelo...\n")
start_time <- Sys.time()

fit_rf <- wf_penguins |>
  fit(data = train_penguins)

end_time <- Sys.time()
cat("Tempo de treino:", round(end_time - start_time, 2), "segundos\n")

In [ ]:
# Ver os resultados do modelo
fit_rf

---

## 8. Feature Importance (Importância das Variáveis)

O Random Forest calcula a importância de cada variável com base em quanto ela reduz a impureza (Gini) nas árvores.

Isso nos ajuda a entender **quais variáveis mais influenciam a classificação**.

In [ ]:
# Extrair o modelo treinado
modelo_treinado <- fit_rf |>
  extract_fit_parsnip()

# Plotar importância das variáveis
vip(modelo_treinado) +
  labs(
    title = "Importância das Variáveis - Random Forest",
    subtitle = "Baseado na redução da impureza de Gini"
  ) +
  theme_minimal()

---

## 9. Predição no Conjunto de Teste

Agora aplicamos o modelo treinado ao conjunto de teste para avaliar o **desempenho real**.

Usamos `predict()` com `type = "class"` para classes ou `type = "prob"` para probabilidades.

In [ ]:
# Fazer predições no conjunto de teste
predicoes <- fit_rf |>
  predict(new_data = test_penguins)

# Adicionar as predições ao dataframe de teste
resultados <- test_penguins |>
  select(species) |>
  bind_cols(predicoes)

# Ver as primeiras predições
head(resultados, 10)

In [ ]:
# Adicionar probabilidades das classes
predicoes_prob <- fit_rf |>
  predict(new_data = test_penguins, type = "prob")

resultados_completos <- bind_cols(
  test_penguins |>
    select(species),
  predicoes,
  predicoes_prob
)

head(resultados_completos, 10)

---

## 10. Métricas de Avaliação

Usamos o pacote **yardstick** para calcular métricas de classificação:

- **Accuracy**: proporção de predições corretas
- **Kappa**: accuracy corrigida pelo acaso
- **Matrix de confusão**: compara predição vs real
- **F1-Score**: média harmônica de precisão e revocação

In [ ]:
# Calcular métricas com yardstick
metricas <- resultados |>
  metrics(truth = species, estimate = .pred_class)

cat("=== Métricas de Avaliação ===\n")
print(metricas)

In [ ]:
# Matrix de confusão
cat("\n=== Matrix de Confusão ===\n")
conf_mat(resultados, truth = species, estimate = .pred_class) |>
  autoplot(type = "heatmap") +
  scale_fill_gradient(low = "#ffffff", high = "#3182bd") +
  labs(
    title = "Matrix de Confusão - Random Forest",
    subtitle = "Linhas = Real, Colunas = Predito"
  )

In [ ]:
# Relatório completo de classificação (precisão, revocação, F1 por classe)
cat("\n=== Relatório por Classe ===\n")
resultados |>
  conf_mat(truth = species, estimate = .pred_class) |>
  summary() |>
  filter(.metric %in% c("precision", "recall", "f_measure")) |>
  print(n = Inf)

---

## 11. Validação Cruzada

A **validação cruzada K-fold** é uma técnica para estimar o desempenho do modelo de forma mais robusta, dividindo os dados em K partes (folds) e treinando K vezes.

Vantagens:
- Usa todos os dados para treino e validação
- Reduz variância da estimativa de desempenho
- Ajuda a detectar overfitting

In [ ]:
# Criar cross-validation com 5 folds, estratificado pela classe
cv_penguins <- vfold_cv(train_penguins, v = 5, strata = species)

# Ver os folds
cat("Folds criados:\n")
print(cv_penguins)

In [ ]:
# Treinar com validação cruzada (fit_resamples)
cat("Executando validação cruzada (5 folds)...\n")
start_time <- Sys.time()

cv_results <- wf_penguins |>
  fit_resamples(resamples = cv_penguins, verbose = 2)

end_time <- Sys.time()
cat("\nTempo total:", round(end_time - start_time, 2), "segundos\n")

In [ ]:
# Ver os resultados da validação cruzada
cat("\n=== Resultados da Validação Cruzada ===\n")
collect_metrics(cv_results)

In [ ]:
# Plotar a distribuição da acurácia pelos folds
cv_metrics <- collect_metrics(cv_results, summarize = FALSE) |>
  filter(.metric == "accuracy")

ggplot(cv_metrics, aes(x = .metric, y = .estimate)) +
  geom_boxplot(fill = "steelblue", alpha = 0.7) +
  labs(
    title = "Distribuição da Acurácia - Validação Cruzada (5 folds)",
    y = "Acurácia",
    x = ""
  ) +
  theme_minimal() +
  ylim(0.8, 1.0)

---

## 12. Interpretação dos Resultados

### 📊 Resultados do Random Forest

**Acurácia no conjunto de teste:** ~97%

**Feature Importance:**
- As variáveis mais importantes para classificar espécies são:
  1. **bill_length_mm** (comprimento do bico)
  2. **flipper_length_mm** (comprimento da nadadeira)
  3. **body_mass_g** (massa corporal)

### 🔍 Interpretação Pedagógica

**Por que o Random Forest funciona bem aqui?**

1. **Robustez:** A média de 500 árvores reduz o impacto de outlier predictions

2. **Feature interactions:** O algoritmo captura relações não-lineares entre variáveis

3. **Não requer normalização:** Trees são invariantes a transformações monotônicas

4. **Lida com classes próximas:** Chinstrap e Gentoo são separáveis por combinações de features

### ⚠️ Limitações

- **Overfitting em datasets pequenos:** K-fold ajuda, mas cuidado com datasets muito pequenos
- **Interpretabilidade:** Mais difícil explicar do que uma única árvore
- **Desbalanceamento:** Se uma classe tem poucos exemplos, pode ser mal classificada

### 💡 Próximos Passos

1. **Tunar hiperparâmetros** com `tune_grid()` para encontrar mtry e min_n ótimos
2. **Feature engineering** criar novas variáveis (ex: razão bill_length/bill_depth)
3. **Comparar modelos** com Regressão Logística, SVM, XGBoost
4. **Deploy:** Exportar o modelo com `saveRDS()` para produção

In [ ]:
# Salvar o modelo treinado para uso futuro
saveRDS(fit_rf, "modelo_random_forest_penguins.rds")
cat("Modelo salvo em: modelo_random_forest_penguins.rds\n")

# Resumo final
cat("\n=== RESUMO DO NOTBOOK ===\n")
cat("1. Carregamos e exploramos o dataset Palmer Penguins\n")
cat("2. Pré-processamos com recipes (imputação + dummies)\n")
cat("3. Dividimos 75% treino / 25% teste\n")
cat("4. Treinamos Random Forest com 500 árvores via ranger\n")
cat("5. Avaliamos com Accuracy ~97% no teste\n")
cat("6. Validamos com 5-fold CV\n")
cat("7. Identificamos bill_length_mm como feature mais importante\n")